# Train, validation and test sets, and cross-validation

A language school wants to predict which students will pass the final exam. You will score Mira's model the
honest way: on rows it never learned from, with 5-fold cross-validation, and with a test set that is used only
once. Same invented students and same code as the lesson.

About 5 minutes. No account or key needed. Run the cells from top to bottom.

## 1. The students

This cell makes the data. The school and its students are invented: a small generator draws them.
Each student has 30 columns: attendance, homework done, mock exam score and 27 other columns that carry no
signal. `passed` is 1 or 0. We draw 120 past students (what the school has) and 5000 "next year" students
(they stand in for the future).

It also defines `make_model(knob)`: a scaler followed by logistic regression with `C=knob`.

**Look for:** 120 past students, 55 of them passed; 5000 next-year students, 2637 passed.

In [1]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, check_cv, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

N_OTHER = 27


def make_students(n, seed):
    rng = np.random.default_rng(seed)
    attendance = np.clip(rng.normal(80, 12, n), 30, 100)
    homework = np.clip(rng.normal(70, 18, n), 0, 100)
    mock_exam = np.clip(rng.normal(62, 15, n), 10, 100)
    other = rng.normal(0, 1, (n, N_OTHER))
    signal = 0.15 * (attendance - 80) + 0.08 * (homework - 70) + 0.18 * (mock_exam - 62)
    p_pass = 1 / (1 + np.exp(-(signal + 0.3)))
    passed = (rng.random(n) < p_pass).astype(int)
    X = np.column_stack([attendance, homework, mock_exam, other])
    return X, passed


def make_model(knob=1.0):
    return make_pipeline(StandardScaler(), LogisticRegression(C=knob, max_iter=5000))


students, passed = make_students(120, seed=7)
next_year, next_passed = make_students(5000, seed=99)
print("past students:     ", len(passed), " passed:", passed.sum(), " columns:", students.shape[1])
print("next-year students:", len(next_passed), " passed:", next_passed.sum())

past students:      120  passed: 55  columns: 30
next-year students: 5000  passed: 2637


## 2. Scored on its own rows

Mira fits the model on all 120 students and scores it on the same 120. Then we score the same model on the
5000 next-year students.

**Look for:** 0.8917 on its own rows, but only 0.7900 on next year's class. The first number measures how well
the model fits the rows it learned from, not how well it predicts new students.

In [2]:
model = make_model().fit(students, passed)
print(f"accuracy on the 120 rows it learned from: {model.score(students, passed):.4f}")
print(f"accuracy on 5000 next-year students:      {model.score(next_year, next_passed):.4f}")

accuracy on the 120 rows it learned from: 0.8917
accuracy on 5000 next-year students:      0.7900


## 3. One hold-out split, ten shuffles

Now Mira trains on 90 students and scores on the other 30, which the model never saw. We repeat this with ten
different shuffles (`random_state` 0 to 9). `stratify=passed` keeps the same share of passes in both parts.

**Look for:** every split is 90 train / 30 test, and the ten test scores go from 0.7000 to 0.8667. Same model,
same data, only the shuffle changed.

In [3]:
holdout = []
for seed in range(10):
    tr_X, te_X, tr_y, te_y = train_test_split(
        students, passed, test_size=0.25, stratify=passed, random_state=seed)
    m = make_model().fit(tr_X, tr_y)
    score = m.score(te_X, te_y)
    holdout.append(score)
    print(f"shuffle {seed}: train {len(tr_y)}, test {len(te_y)}, test accuracy {score:.4f}")
print(f"lowest {min(holdout):.4f}, highest {max(holdout):.4f}")

shuffle 0: train 90, test 30, test accuracy 0.8333
shuffle 1: train 90, test 30, test accuracy 0.8000
shuffle 2: train 90, test 30, test accuracy 0.7667
shuffle 3: train 90, test 30, test accuracy 0.7667
shuffle 4: train 90, test 30, test accuracy 0.8667
shuffle 5: train 90, test 30, test accuracy 0.7000
shuffle 6: train 90, test 30, test accuracy 0.7333
shuffle 7: train 90, test 30, test accuracy 0.8000
shuffle 8: train 90, test 30, test accuracy 0.8333
shuffle 9: train 90, test 30, test accuracy 0.8000
lowest 0.7000, highest 0.8667


## 4. The first 20 students, dealt into 5 folds

This is the by-hand screen of the lesson. `StratifiedKFold(n_splits=5)` deals the first 20 students into 5
folds of 4. The names are invented (the data has no names).

**Look for:** fold A is Amira, Ben, Chloe, Dev. Each fold holds 4 students, and the 6 students who passed are
spread over the folds (1 or 2 per fold), so no fold is all fails.

In [4]:
NAMES = ["Amira", "Ben", "Chloe", "Dev", "Elif", "Farid", "Grace", "Hugo", "Ines", "Jonas",
         "Kaya", "Leo", "Maya", "Nils", "Omar", "Priya", "Quinn", "Rosa", "Sami", "Tara"]
small_y = passed[:20]
splitter = StratifiedKFold(n_splits=5)
for k, (train_rows, scored_rows) in enumerate(splitter.split(np.zeros((20, 1)), small_y)):
    who = [f"{NAMES[i]} ({'pass' if small_y[i] else 'fail'})" for i in scored_rows]
    print(f"fold {'ABCDE'[k]}: " + ", ".join(who))

fold A: Amira (fail), Ben (pass), Chloe (fail), Dev (fail)
fold B: Elif (fail), Farid (pass), Grace (fail), Hugo (fail)
fold C: Ines (fail), Jonas (fail), Kaya (fail), Maya (pass)
fold D: Leo (fail), Nils (fail), Omar (fail), Priya (pass)
fold E: Quinn (pass), Rosa (pass), Sami (fail), Tara (fail)


## 5. 5-fold cross-validation on all 120

`cross_val_score(..., cv=5)` trains 5 models. Each one learns from 96 students and is scored on the other 24.
Every student is scored exactly once.

**Look for:** 5 fold scores (0.8333, 0.6667, 0.8333, 0.8333, 0.6667), mean 0.7667 and standard deviation
0.0816. The mean is close to next year's 0.7900. With a classifier, `cv=5` means `StratifiedKFold`, and it does
not shuffle.

In [5]:
fold_scores = cross_val_score(make_model(), students, passed, cv=5)
print("splitter used for cv=5:", type(check_cv(5, passed, classifier=True)).__name__)
for k, (tr, te) in enumerate(StratifiedKFold(n_splits=5).split(students, passed)):
    print(f"fold {k + 1}: train {len(tr)}, scored {len(te)} ({passed[te].sum()} passed), accuracy {fold_scores[k]:.4f}")
print(f"mean {fold_scores.mean():.4f}, standard deviation {fold_scores.std():.4f}")

splitter used for cv=5: StratifiedKFold
fold 1: train 96, scored 24 (11 passed), accuracy 0.8333
fold 2: train 96, scored 24 (11 passed), accuracy 0.6667
fold 3: train 96, scored 24 (11 passed), accuracy 0.8333
fold 4: train 96, scored 24 (11 passed), accuracy 0.8333
fold 5: train 96, scored 24 (11 passed), accuracy 0.6667
mean 0.7667, standard deviation 0.0816


## 6. Choosing with the test rows (the mistake)

Logistic regression has a setting `C`. Here it is just a knob with 4 positions. Mira holds back 30 test
students, trains one model per knob on the other 90, and scores each on the same 30 test students. We also
print each model's real accuracy on next year's class, which Mira cannot see.

**Look for:** C = 0.1 and C = 1 tie at 0.8333 on the test rows. Picking the best test score and reporting it
is too rosy: the next-year accuracy of those models is lower (0.7746 and 0.7848).

In [6]:
KNOBS = [0.01, 0.1, 1, 10]
dev_X, test_X, dev_y, test_y = train_test_split(
    students, passed, test_size=0.25, stratify=passed, random_state=0)
print("dev rows:", len(dev_y), " test rows:", len(test_y))
for knob in KNOBS:
    m = make_model(knob).fit(dev_X, dev_y)
    print(f"C = {knob:<5} test accuracy {m.score(test_X, test_y):.4f}   next year {m.score(next_year, next_passed):.4f}")

dev rows: 90  test rows: 30
C = 0.01  test accuracy 0.8000   next year 0.7158
C = 0.1   test accuracy 0.8333   next year 0.7746
C = 1     test accuracy 0.8333   next year 0.7848
C = 10    test accuracy 0.7333   next year 0.7744


## 7. The honest procedure

`honest_score` splits once into 90 dev and 30 test students. It chooses the knob with 5-fold cross-validation
on the dev rows only (they play the validation set). Then it trains the chosen model on all 90 dev rows and
scores the test rows once, at the very end. The `print` inside the loop is added so you can follow it.

**Look for:** CV means 0.6667, 0.7556, 0.7444, 0.7444. The function returns `(0.1, 0.7556, 0.8333)`: the
chosen knob, its CV score, and the test score.

In [7]:
def honest_score(students, passed):
    dev_X, test_X, dev_y, test_y = train_test_split(
        students, passed, test_size=0.25, stratify=passed, random_state=0)
    best_knob, best_cv = None, 0
    for knob in KNOBS:
        model = make_model(knob)
        cv = cross_val_score(model, dev_X, dev_y, cv=5).mean()
        print(f"C = {knob:<5} CV mean on the dev rows {cv:.4f}")
        if cv > best_cv:
            best_knob, best_cv = knob, cv
    final = make_model(best_knob).fit(dev_X, dev_y)
    return best_knob, best_cv, final.score(test_X, test_y)


best_knob, best_cv, test_acc = honest_score(students, passed)
print("returned:", (best_knob, round(float(best_cv), 4), round(float(test_acc), 4)))

C = 0.01  CV mean on the dev rows 0.6667
C = 0.1   CV mean on the dev rows 0.7556


C = 1     CV mean on the dev rows 0.7444


C = 10    CV mean on the dev rows 0.7444
returned: (0.1, 0.7556, 0.8333)


## 8. Why it matters: 200 shuffles

One split can be lucky either way. So we repeat Mira's experiment with 200 different shuffles. For each, we
compare the reported score with that model's real accuracy on next year's class. "Peek" picks the knob with
the best test score. "Honest" picks with CV on the dev rows and scores the test rows once.
This cell trains a few thousand small models; it takes about half a minute.

**Look for:** peek is on average 4.25 points too high and too high in 75.5% of shuffles. Honest is on average
0.53 points too low and too high in 44% of shuffles: not tilted.

In [8]:
gap_peek, gap_honest = [], []
for seed in range(200):
    dX, tX, dy, ty = train_test_split(students, passed, test_size=0.25, stratify=passed, random_state=seed)
    fitted = {k: make_model(k).fit(dX, dy) for k in KNOBS}
    test_scores = {k: fitted[k].score(tX, ty) for k in KNOBS}
    peek = max(KNOBS, key=lambda k: test_scores[k])
    gap_peek.append(test_scores[peek] - fitted[peek].score(next_year, next_passed))
    cvs = {k: cross_val_score(make_model(k), dX, dy, cv=5).mean() for k in KNOBS}
    pick = max(KNOBS, key=lambda k: cvs[k])
    gap_honest.append(test_scores[pick] - fitted[pick].score(next_year, next_passed))

print(f"peek:   mean gap {np.mean(gap_peek):+.4f}, above the truth in {np.mean(np.array(gap_peek) > 0):.1%} of shuffles")
print(f"honest: mean gap {np.mean(gap_honest):+.4f}, above the truth in {np.mean(np.array(gap_honest) > 0):.1%} of shuffles")

peek:   mean gap +0.0425, above the truth in 75.5% of shuffles
honest: mean gap -0.0053, above the truth in 44.0% of shuffles


## Now change it

Three small experiments. Each cell has one line marked `# <- change`. Edit it, run the cell, and compare.

### Experiment 1: more past students

With 120 students, a 30-row test set swings a lot. Change `N_PAST` to `1000` and run again.

**What should change:** the gap between the lowest and highest of the ten hold-out scores gets much smaller,
because each test set now has 250 students, and one student moves the score far less.

In [9]:
N_PAST = 120  # <- change: try 1000
X_more, y_more = make_students(N_PAST, seed=7)
scores = []
for seed in range(10):
    tr_X, te_X, tr_y, te_y = train_test_split(X_more, y_more, test_size=0.25, stratify=y_more, random_state=seed)
    scores.append(make_model().fit(tr_X, tr_y).score(te_X, te_y))
print(f"{N_PAST} students, test rows per split: {len(te_y)}")
print(f"ten hold-out scores: lowest {min(scores):.4f}, highest {max(scores):.4f}, range {max(scores) - min(scores):.4f}")

120 students, test rows per split: 30
ten hold-out scores: lowest 0.7000, highest 0.8667, range 0.1667


### Experiment 2: more folds

Change `cv=5` to `cv=10`.

**What should change:** you get 10 fold scores, and each one rests on only 12 students. So the lowest and
highest fold scores move further apart (one student is now 8.3 points), while the mean stays in the same area.
Each model trains on 108 students instead of 96.

In [10]:
scores = cross_val_score(make_model(), students, passed, cv=5)  # <- change: try cv=10
print("number of folds:", len(scores))
print("fold scores:", np.round(scores, 4))
print(f"lowest {scores.min():.4f}, highest {scores.max():.4f}")
print(f"mean {scores.mean():.4f}, standard deviation {scores.std():.4f}")

number of folds: 5
fold scores: [0.8333 0.6667 0.8333 0.8333 0.6667]
lowest 0.6667, highest 0.8333
mean 0.7667, standard deviation 0.0816


### Experiment 3: shuffle the folds

`cv=5` does not shuffle. Here we pass a splitter that does. Change `random_state=0` to `1`, `2`, `3`.

**What should change:** the five fold scores change with every seed, and the mean moves a little too. The mean
moves much less (about 4 points over these seeds) than a single hold-out score did in step 3 (17 points). That is why you report the mean and the spread.

In [11]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)  # <- change: try 1, 2, 3
scores = cross_val_score(make_model(), students, passed, cv=folds)
print("fold scores:", np.round(scores, 4))
print(f"mean {scores.mean():.4f}, standard deviation {scores.std():.4f}")

fold scores:

 [0.625  0.7917 0.7083 0.875  0.6667]
mean 0.7333, standard deviation 0.0898


## What you should see

- The accuracy on the 120 training students prints 0.8917, higher than every fold score.
- Ten 75/25 shuffles give test scores between 0.7000 and 0.8667.
- `cross_val_score(..., cv=5)` returns 5 numbers, each from 24 scored students; their mean is 0.7667.
- `honest_score` chooses `C = 0.1` and returns the test score 0.8333, measured once, at the end.
- Over 200 shuffles, picking the knob on the test rows is too high on average (+0.0425); the honest procedure is not tilted (-0.0053).